# FFT-accelerated beta-2 (L2) divergence experiments

> **Main beta-2/$L^2$ notebook.** This is the canonical location for numerical experiments based on the beta-2 divergence. The experiments use the FFT implementation below, with the analytic Gaussian formula retained for validation.

For the density-power-divergence convention, beta equal to 2 gives half the squared L2 distance:

$$
D_2(p,q)=\frac{1}{2}\int_{-\infty}^{\infty}(p(x)-q(x))^2\,\mathrm{d}x.
$$

For Gaussian KDEs based on the same sample, define the exact Gram matrix

$$
G_{ij}=\int \widehat f_{h_i}(x)\widehat f_{h_j}(x)\,\mathrm{d}x.
$$

Then

$$
D_2(\widehat f_{h_i},\widehat f_{h_j})
=\frac{1}{2}\left(G_{ii}+G_{jj}-2G_{ij}\right).
$$

The analytic Gaussian-KDE formula is exact but costs $O(N^2n^2)$. The experiments below instead evaluate the whole KDE family once by FFT and assemble $G$ with the matrix product $G \approx \Delta x\,FF^T$. The analytic helper is retained for the small validation check below. Control the FFT approximation with `beta2_half_width` and `beta2_spatial_points`.

All exact numerical Perron computations use the centralized stabilized Lanczos solver in `kde_tools.perron`. Doob masses are derived from that same vector.


In [ ]:
from __future__ import annotations

import importlib
from pathlib import Path

import matplotlib.pyplot as plt                                                                     ### PLOTTING
import numpy as np
import plotly.graph_objects as go                                                                   ### PLOTTING
from plotly.subplots import make_subplots                                                           ### PLOTTING

from kde_tools.perron import (
    PERRON_SOLVER_VERSION,
    doob_mass_from_perron,
    perron_eigenpair_from_divergence,
    perron_peak_index,
    perron_vector_from_divergence,
)
from kde_tools.pdf_export import enable_automatic_pdf_saving                                        ### PLOTTING
from kde_tools.bandwidths import (
    bandwidth_grid_linear_in_psi_beta,
    gaussian_mise_oracle_bandwidth,
    nearest_bandwidth_grid_index,
)
from kde_tools.density import exact_oracle_ise_values
from kde_tools.fourier import (
    beta2_divergence_matrix_fft,
    kde_family_fft,
)
import kde_tools.plotting_functions as kde_plotting_functions                                       ### PLOTTING
import kde_tools.similarities as kde_similarities

# Reload helpers that may have changed since this kernel was started.
importlib.reload(kde_plotting_functions)
importlib.reload(kde_similarities)
from kde_tools.plotting_functions import (                                                          ### PLOTTING
    plot_bandwidth_distributions,                                                                   ### PLOTTING
    plot_perron_vector,                                                                             ### PLOTTING
)                                                                                                   ### PLOTTING
from kde_tools.similarities import (
    beta2_divergence_matrix,
    first_order_perron_vector,
    population_gaussian_beta2_divergence_matrix,
)

np.set_printoptions(precision=5, suppress=True)                                                     ### REPORTING
seed = 20260818

project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "Code" / "Code_kde").is_dir()
        and (candidate / "Abschlussarbeit_2021" / "LaTeX").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate the MASTERARBEIT project root.")

kde_thesis_figure_dir = (
    project_root / "Code" / "Code_kde" / "outputs" / "images" / "thesis"
)
kde_thesis_figure_dir.mkdir(parents=True, exist_ok=True)                                            ### PLOTTING

def save_thesis_plotly_figure(figure, filename: str) -> None:
    from plotly.io import kaleido                                                                   ### PLOTTING
    kaleido.scope.mathjax = None
    figure.write_image(str(kde_thesis_figure_dir / filename), scale=1)                              ### PLOTTING

pdf_output_directory = enable_automatic_pdf_saving(                                                 ### PLOTTING
    "spectral_scale_selection_kde"                                                                  ### PLOTTING
)                                                                                                   ### PLOTTING

print(f"Perron solver: {PERRON_SOLVER_VERSION}")


## 1. Example stationary distribution on a grid linear in $\psi_\beta$

Choose the sample size, number of bandwidth candidates, bandwidth interval, grid parameter $\beta$, and similarity scales below. The bandwidth candidates are uniformly spaced in $\psi_\beta(h)$. For $\beta=2$, this means uniform spacing in $\log h$, so the grid is geometric. The same fixed sample and grid are used for four Perron curves at $\lambda\in\{1,10,100,1000\}$. Dots show the Perron-vector components at the grid bandwidths, and a dashed vertical line in the corresponding curve colour marks each maximum. The blue dashed line marks the Gaussian MISE-oracle bandwidth, and the purple dashed line marks the sample-specific ISE-minimizing grid bandwidth.


In [ ]:
stationary_example_seed = seed + 5
stationary_example_n = 200
stationary_example_N = 191
stationary_example_beta = 2.0
stationary_example_h_min = 0.01
stationary_example_h_max = 2.0
stationary_example_lambdas = np.array([1.0, 10.0, 100.0, 1000.0])
stationary_example_half_width = 12.0
stationary_example_spatial_points = 8192

if stationary_example_n < 1:
    raise ValueError("stationary_example_n must be positive.")
if stationary_example_N < 2:
    raise ValueError("stationary_example_N must be at least two.")
if not 0.0 < stationary_example_h_min < stationary_example_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if np.any(stationary_example_lambdas <= 0.0):
    raise ValueError("stationary_example_lambdas must be positive.")
if stationary_example_half_width <= 0.0:
    raise ValueError("stationary_example_half_width must be positive.")
if (
    stationary_example_spatial_points < 2
    or stationary_example_spatial_points % 2
):
    raise ValueError(
        "stationary_example_spatial_points must be a positive even integer."
    )

stationary_example_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=stationary_example_h_min,
    h_max=stationary_example_h_max,
    beta=stationary_example_beta,
    grid_size=stationary_example_N,
)
stationary_example_rng = np.random.default_rng(stationary_example_seed)
stationary_example_samples = stationary_example_rng.normal(
    size=stationary_example_n
)
stationary_example_divergence = beta2_divergence_matrix_fft(
    samples=stationary_example_samples,
    h_grid=stationary_example_grid,
    half_width=stationary_example_half_width,
    spatial_points=stationary_example_spatial_points,
)
stationary_example_similarities = {}
stationary_example_perron_vectors = {}
stationary_example_spectral_gaps = {}
stationary_example_peak_indices = {}
stationary_example_peak_bandwidths = {}

for current_lambda in stationary_example_lambdas:
    current_similarity = np.exp(
        -current_lambda * stationary_example_divergence
    )
    np.fill_diagonal(current_similarity, 1.0)
    current_perron_result = perron_eigenpair_from_divergence(
        stationary_example_divergence, current_lambda
    )
    current_perron = current_perron_result.vector
    current_peak_index = int(perron_peak_index(current_perron))

    stationary_example_similarities[current_lambda] = current_similarity
    stationary_example_perron_vectors[current_lambda] = current_perron
    stationary_example_spectral_gaps[current_lambda] = float(
        np.exp(-current_lambda * current_perron_result.minimum_divergence)
        * current_perron_result.spectral_gap
        / (1.0 + np.exp(-current_lambda * current_perron_result.minimum_divergence)
           * current_perron_result.eigenvalue)
    )
    stationary_example_peak_indices[current_lambda] = current_peak_index
    stationary_example_peak_bandwidths[current_lambda] = float(
        stationary_example_grid[current_peak_index]
    )

stationary_example_reference_lambda = float(
    stationary_example_lambdas[0]
)
stationary_example_similarity = stationary_example_similarities[
    stationary_example_reference_lambda
]
stationary_example_perron = stationary_example_perron_vectors[
    stationary_example_reference_lambda
]
stationary_example_mass = doob_mass_from_perron(
    stationary_example_perron
)
stationary_example_peak_index = stationary_example_peak_indices[
    stationary_example_reference_lambda
]
stationary_example_peak_h = stationary_example_peak_bandwidths[
    stationary_example_reference_lambda
]
stationary_example_spectral_gap = stationary_example_spectral_gaps[
    stationary_example_reference_lambda
]
stationary_example_oracle_h = gaussian_mise_oracle_bandwidth(
    stationary_example_n
)
stationary_example_ise_values = exact_oracle_ise_values(
    samples=stationary_example_samples,
    h_grid=stationary_example_grid,
    sigma2=1.0,
)
stationary_example_ise_index = int(
    np.argmin(stationary_example_ise_values)
)
stationary_example_ise_h = float(
    stationary_example_grid[stationary_example_ise_index]
)
stationary_example_target_roughness = 3.0 / (8.0 * np.sqrt(np.pi))
stationary_example_h_speed = (
    3.0
    / (
        32.0
        * np.sqrt(np.pi)
        * stationary_example_n
        * stationary_example_target_roughness
    )
) ** (1.0 / 5.0)
stationary_example_log_h_min = np.log10(stationary_example_h_min)
stationary_example_log_h_max = np.log10(stationary_example_h_max)
stationary_example_log_h_span = (
    stationary_example_log_h_max - stationary_example_log_h_min
)
stationary_example_h_speed_paper_x = (
    np.log10(stationary_example_h_speed)
    - stationary_example_log_h_min
) / stationary_example_log_h_span
stationary_example_h_speed_index = (
    stationary_example_h_speed_paper_x
    * (stationary_example_N - 1)
)

fig, ax = plt.subplots(figsize=(10.5, 5.5))                                                         ### PLOTTING
stationary_example_curve_colors = (
    "#E69F00",  # orange
    "#009E73",  # bluish green
    "#D55E00",  # vermillion
    "#222222",  # near black
)
if len(stationary_example_curve_colors) != len(
    stationary_example_lambdas
):
    raise ValueError("Provide one curve colour per lambda value.")

for current_lambda, current_color in zip(
    stationary_example_lambdas,
    stationary_example_curve_colors,
):
    current_perron = stationary_example_perron_vectors[current_lambda]
    ax.plot(                                                                                        ### PLOTTING
        stationary_example_grid,                                                                    ### PLOTTING
        current_perron,                                                                             ### PLOTTING
        color=current_color,                                                                        ### PLOTTING
        linewidth=2.0,                                                                              ### PLOTTING
        marker="o",                                                                                 ### PLOTTING
        markersize=3.5,                                                                             ### PLOTTING
        label=rf"$\lambda={current_lambda:g}$",                                                     ### PLOTTING
    )                                                                                               ### PLOTTING

ax.set_xscale("log")                                                                                ### PLOTTING
ax.set_xlabel("Bandwidth h")                                                                        ### PLOTTING
ax.set_ylabel("Perron-vector component")                                                            ### PLOTTING
ax.grid(alpha=0.3)                                                                                  ### PLOTTING
for current_lambda, current_color in zip(                                                           ### PLOTTING
    stationary_example_lambdas,                                                                     ### PLOTTING
    stationary_example_curve_colors,                                                                ### PLOTTING
):                                                                                                  ### PLOTTING
    ax.axvline(                                                                                     ### PLOTTING
        stationary_example_peak_bandwidths[current_lambda],                                         ### PLOTTING
        color=current_color,                                                                        ### PLOTTING
        linestyle="--",                                                                             ### PLOTTING
        linewidth=1.2,                                                                              ### PLOTTING
        alpha=0.75,                                                                                 ### PLOTTING
        label="_nolegend_",                                                                         ### PLOTTING
    )                                                                                               ### PLOTTING
ax.axvline(                                                                                         ### PLOTTING
    stationary_example_oracle_h,                                                                    ### PLOTTING
    color="tab:blue",                                                                               ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    label=f"Gaussian MISE oracle: h={stationary_example_oracle_h:.4g}",                             ### PLOTTING
)                                                                                                   ### PLOTTING
ax.axvline(                                                                                         ### PLOTTING
    stationary_example_ise_h,                                                                       ### PLOTTING
    color="tab:purple",                                                                             ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    label=f"sample ISE minimum: h={stationary_example_ise_h:.4g}",                                  ### PLOTTING
)                                                                                                   ### PLOTTING
ax.legend(ncol=2, fontsize=8.5)                                                                     ### PLOTTING
plt.tight_layout()                                                                                  ### PLOTTING
fig.savefig(                                                                                        ### PLOTTING
    kde_thesis_figure_dir / "stationary_distribution_plot_001.pdf",                                 ### PLOTTING
    dpi=100,                                                                                        ### PLOTTING
)                                                                                                   ### PLOTTING
plt.show()                                                                                          ### PLOTTING


### Similarity scale versus chosen bandwidth

For the same fixed sample as Figure 3.1, the plot below uses $N=201$ bandwidths linear in $\psi_\beta$, varies the similarity scale $\lambda$ in $S_{ij}=\exp(-\lambda D_{2,ij})$, and records the bandwidth at the maximum of the Doob stationary mass. The blue dashed horizontal line marks the Gaussian MISE-oracle bandwidth, and the purple dashed horizontal line marks the sample-specific ISE-minimizing grid bandwidth.

The two red dashed horizontal lines mark the bandwidths in the closest adjacent pair on this grid, found by minimizing the adjacent beta-2 divergences.


In [ ]:
stationary_example_lambda_values = np.geomspace(1e-2, 1e4, 101)
stationary_example_lambda_grid_N = 201
stationary_example_lambda_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=stationary_example_h_min,
    h_max=stationary_example_h_max,
    beta=stationary_example_beta,
    grid_size=stationary_example_lambda_grid_N,
)
stationary_example_log_lambda_min = np.log10(
    stationary_example_lambda_values[0]
)
stationary_example_log_lambda_max = np.log10(
    stationary_example_lambda_values[-1]
)
stationary_example_lambda_ise_values = exact_oracle_ise_values(
    samples=stationary_example_samples,
    h_grid=stationary_example_lambda_grid,
    sigma2=1.0,
)
stationary_example_lambda_ise_index = int(
    np.argmin(stationary_example_lambda_ise_values)
)
stationary_example_lambda_ise_h = float(
    stationary_example_lambda_grid[
        stationary_example_lambda_ise_index
    ]
)
stationary_example_lambda_divergence = beta2_divergence_matrix_fft(
    samples=stationary_example_samples,
    h_grid=stationary_example_lambda_grid,
    half_width=stationary_example_half_width,
    spatial_points=stationary_example_spatial_points,
)
# Identify the closest adjacent pair on the same grid used for selection.
stationary_example_lambda_closest_pair_left_index = int(
    np.argmin(np.diag(stationary_example_lambda_divergence, k=1))
)
stationary_example_lambda_closest_pair_indices = (
    stationary_example_lambda_closest_pair_left_index,
    stationary_example_lambda_closest_pair_left_index + 1,
)
stationary_example_lambda_closest_pair_bandwidths = (
    stationary_example_lambda_grid[
        list(stationary_example_lambda_closest_pair_indices)
    ]
)

stationary_example_lambda_selected_indices = np.empty(
    len(stationary_example_lambda_values),
    dtype=int,
)

for index, current_lambda in enumerate(
    stationary_example_lambda_values
):
    current_perron = perron_vector_from_divergence(
        stationary_example_lambda_divergence, current_lambda
    )
    current_mass = doob_mass_from_perron(current_perron)
    stationary_example_lambda_selected_indices[index] = int(
        perron_peak_index(current_perron)
    )

stationary_example_lambda_selected_bandwidths = (
    stationary_example_lambda_grid[
        stationary_example_lambda_selected_indices
    ]
)

fig, ax = plt.subplots(figsize=(10.5, 5.5))                                                         ### PLOTTING
ax.step(                                                                                            ### PLOTTING
    stationary_example_lambda_values,                                                               ### PLOTTING
    stationary_example_lambda_selected_bandwidths,                                                  ### PLOTTING
    where="post",                                                                                   ### PLOTTING
    color="#222222",                                                                                ### PLOTTING
    linewidth=2.0,                                                                                  ### PLOTTING
    marker="o",                                                                                     ### PLOTTING
    markersize=3.5,                                                                                 ### PLOTTING
    label=(                                                                                         ### PLOTTING
        "spectral selection "                                                                       ### PLOTTING
        rf"($N={stationary_example_lambda_grid_N}$)"                                                ### PLOTTING
    ),                                                                                              ### PLOTTING
)                                                                                                   ### PLOTTING
ax.axhline(                                                                                         ### PLOTTING
    stationary_example_oracle_h,                                                                    ### PLOTTING
    color="tab:blue",                                                                               ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    linewidth=1.8,                                                                                  ### PLOTTING
    label=f"Gaussian MISE oracle: h={stationary_example_oracle_h:.4g}",                             ### PLOTTING
)                                                                                                   ### PLOTTING
ax.axhline(                                                                                         ### PLOTTING
    stationary_example_lambda_ise_h,                                                                ### PLOTTING
    color="tab:purple",                                                                             ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    linewidth=1.8,                                                                                  ### PLOTTING
    label=(                                                                                         ### PLOTTING
        "sample ISE minimum: "                                                                      ### PLOTTING
        f"h={stationary_example_lambda_ise_h:.4g}"                                                  ### PLOTTING
    ),                                                                                              ### PLOTTING
)                                                                                                   ### PLOTTING
for grid_index, bandwidth in zip(                                                                  ### PLOTTING
    stationary_example_lambda_closest_pair_indices,                                                 ### PLOTTING
    stationary_example_lambda_closest_pair_bandwidths,                                              ### PLOTTING
):                                                                                                  ### PLOTTING
    ax.axhline(                                                                                     ### PLOTTING
        bandwidth,                                                                                  ### PLOTTING
        color="tab:red",                                                                            ### PLOTTING
        linestyle="--",                                                                             ### PLOTTING
        linewidth=0.9,                                                                              ### PLOTTING
        zorder=1.8,                                                                                 ### PLOTTING
        label=(                                                                                     ### PLOTTING
            "closest pair: "                                                                        ### PLOTTING
            rf"$h_{{{grid_index + 1}}}={bandwidth:.6f}$"                                             ### PLOTTING
        ),                                                                                          ### PLOTTING
    )                                                                                               ### PLOTTING
ax.set_xscale("log")                                                                                ### PLOTTING
ax.set_yscale("log")                                                                                ### PLOTTING
ax.set_xlim(                                                                                        ### PLOTTING
    stationary_example_lambda_values[0],                                                            ### PLOTTING
    stationary_example_lambda_values[-1],                                                           ### PLOTTING
)                                                                                                   ### PLOTTING
ax.set_ylim(                                                                                        ### PLOTTING
    stationary_example_h_min,                                                                       ### PLOTTING
    stationary_example_h_max,                                                                       ### PLOTTING
)                                                                                                   ### PLOTTING
ax.set_xlabel(r"similarity scale $\lambda$")                                                        ### PLOTTING
ax.set_ylabel("chosen bandwidth h")                                                                 ### PLOTTING
ax.grid(alpha=0.3)                                                                                  ### PLOTTING
ax.legend(fontsize=8.5)                                                                             ### PLOTTING
plt.tight_layout()                                                                                  ### PLOTTING
fig.savefig(                                                                                        ### PLOTTING
    kde_thesis_figure_dir                                                                           ### PLOTTING
    / "chosen_bandwidth_vs_lambda_M1_fixed_sample.pdf",                                             ### PLOTTING
    dpi=100,                                                                                        ### PLOTTING
)                                                                                                   ### PLOTTING
plt.show()                                                                                          ### PLOTTING


### Monte Carlo median selection and spread across samples

For similarity scales from $10^{-2}$ through $2\times10^4$, this experiment draws $M=200$ independent samples of size $n$, records the chosen bandwidth-grid index, and plots its median. The shaded band is the interquartile range from the 25th to the 75th percentile. Each repetition reuses its sample across all $\lambda$ values, so changes along a repetition are paired. The gray dashed horizontal line marks the theoretical speed bandwidth.


In [ ]:
stationary_example_mc_M = 1
stationary_example_mc_seed = stationary_example_seed + 1000
stationary_example_mc_lambda_values = np.geomspace(
    1e-2, 2e4, 106
)
stationary_example_mc_log_lambda_min = np.log10(
    stationary_example_mc_lambda_values[0]
)
stationary_example_mc_log_lambda_max = np.log10(
    stationary_example_mc_lambda_values[-1]
)

if stationary_example_mc_M < 1:
    raise ValueError("stationary_example_mc_M must be positive.")

stationary_example_mc_rng = np.random.default_rng(
    stationary_example_mc_seed
)
stationary_example_mc_selected_indices = np.empty(
    (
        stationary_example_mc_M,
        len(stationary_example_mc_lambda_values),
    ),
    dtype=np.int16,
)

for repetition in range(stationary_example_mc_M):
    current_samples = stationary_example_mc_rng.normal(
        size=stationary_example_n
    )
    current_divergence = beta2_divergence_matrix_fft(
        samples=current_samples,
        h_grid=stationary_example_grid,
        half_width=stationary_example_half_width,
        spatial_points=stationary_example_spatial_points,
    )

    for lambda_index, current_lambda in enumerate(
        stationary_example_mc_lambda_values
    ):
        current_perron = perron_vector_from_divergence(
            current_divergence, current_lambda
        )
        current_mass = doob_mass_from_perron(current_perron)
        stationary_example_mc_selected_indices[
            repetition, lambda_index
        ] = int(perron_peak_index(current_perron))

    if (repetition + 1) % 10 == 0:                                                                  ### REPORTING
        print(                                                                                      ### REPORTING
            f"completed {repetition + 1}/"                                                          ### REPORTING
            f"{stationary_example_mc_M} repetitions"                                                ### REPORTING
        )                                                                                           ### REPORTING

(
    stationary_example_mc_q25_indices,
    stationary_example_mc_median_indices,
    stationary_example_mc_q75_indices,
) = np.quantile(
    stationary_example_mc_selected_indices,
    [0.25, 0.50, 0.75],
    axis=0,
)

stationary_example_index_grid = np.arange(
    stationary_example_N, dtype=float
)
stationary_example_log_bandwidth_grid = np.log(
    stationary_example_grid
)

def stationary_example_bandwidth_at_index(indices):
    return np.exp(
        np.interp(
            indices,
            stationary_example_index_grid,
            stationary_example_log_bandwidth_grid,
        )
    )

stationary_example_mc_q25_bandwidths = (
    stationary_example_bandwidth_at_index(
        stationary_example_mc_q25_indices
    )
)
stationary_example_mc_median_bandwidths = (
    stationary_example_bandwidth_at_index(
        stationary_example_mc_median_indices
    )
)
stationary_example_mc_q75_bandwidths = (
    stationary_example_bandwidth_at_index(
        stationary_example_mc_q75_indices
    )
)


In [ ]:
stationary_example_mc_oracle_index = float(
    np.interp(
        np.log(stationary_example_oracle_h),
        stationary_example_log_bandwidth_grid,
        stationary_example_index_grid,
    )
)
stationary_example_mc_oracle_paper_y = (
    stationary_example_mc_oracle_index + 0.5
) / stationary_example_N
stationary_example_mc_h_speed_paper_y = (
    stationary_example_h_speed_index + 0.5
) / stationary_example_N
stationary_example_mc_tick_indices = np.unique(
    np.linspace(
        0, stationary_example_N - 1, 7, dtype=int
    )
)
stationary_example_mc_tick_labels = [
    f"{index}<br>h={stationary_example_grid[index]:.3g}"
    for index in stationary_example_mc_tick_indices
]

stationary_example_mc_figure = go.Figure()                                                          ### PLOTTING
stationary_example_mc_figure.add_trace(                                                             ### PLOTTING
    go.Scatter(                                                                                     ### PLOTTING
        x=stationary_example_mc_lambda_values,                                                      ### PLOTTING
        y=stationary_example_mc_q25_indices,                                                        ### PLOTTING
        customdata=stationary_example_mc_q25_bandwidths,                                            ### PLOTTING
        mode="lines",                                                                               ### PLOTTING
        line=dict(color="rgba(31,119,180,0)", shape="hv"),                                          ### PLOTTING
        name="25th percentile",                                                                     ### PLOTTING
        showlegend=False,                                                                           ### PLOTTING
        hovertemplate=(                                                                             ### PLOTTING
            "λ=%{x:.6g}"                                                                            ### PLOTTING
            "<br>25th-percentile index=%{y:.3g}"                                                    ### PLOTTING
            "<br>corresponding h=%{customdata:.6g}<extra></extra>"                                  ### PLOTTING
        ),                                                                                          ### PLOTTING
    )                                                                                               ### PLOTTING
)                                                                                                   ### PLOTTING
stationary_example_mc_figure.add_trace(                                                             ### PLOTTING
    go.Scatter(                                                                                     ### PLOTTING
        x=stationary_example_mc_lambda_values,                                                      ### PLOTTING
        y=stationary_example_mc_q75_indices,                                                        ### PLOTTING
        customdata=stationary_example_mc_q75_bandwidths,                                            ### PLOTTING
        mode="lines",                                                                               ### PLOTTING
        line=dict(color="rgba(31,119,180,0)", shape="hv"),                                          ### PLOTTING
        fill="tonexty",                                                                             ### PLOTTING
        fillcolor="rgba(31,119,180,0.22)",                                                          ### PLOTTING
        name="interquartile range",                                                                 ### PLOTTING
        hovertemplate=(                                                                             ### PLOTTING
            "λ=%{x:.6g}"                                                                            ### PLOTTING
            "<br>75th-percentile index=%{y:.3g}"                                                    ### PLOTTING
            "<br>corresponding h=%{customdata:.6g}<extra></extra>"                                  ### PLOTTING
        ),                                                                                          ### PLOTTING
    )                                                                                               ### PLOTTING
)                                                                                                   ### PLOTTING
stationary_example_mc_figure.add_trace(                                                             ### PLOTTING
    go.Scatter(                                                                                     ### PLOTTING
        x=stationary_example_mc_lambda_values,                                                      ### PLOTTING
        y=stationary_example_mc_median_indices,                                                     ### PLOTTING
        customdata=stationary_example_mc_median_bandwidths,                                         ### PLOTTING
        mode="lines+markers",                                                                       ### PLOTTING
        line=dict(color="#1f77b4", width=2.5, shape="hv"),                                          ### PLOTTING
        marker=dict(size=5),                                                                        ### PLOTTING
        name="median selected index",                                                               ### PLOTTING
        hovertemplate=(                                                                             ### PLOTTING
            "λ=%{x:.6g}"                                                                            ### PLOTTING
            "<br>median index=%{y:.3g}"                                                             ### PLOTTING
            "<br>corresponding h=%{customdata:.6g}<extra></extra>"                                  ### PLOTTING
        ),                                                                                          ### PLOTTING
    )                                                                                               ### PLOTTING
)                                                                                                   ### PLOTTING
stationary_example_mc_figure.add_hline(                                                             ### PLOTTING
    y=stationary_example_mc_oracle_index,                                                           ### PLOTTING
    line=dict(color="#d62728", width=2, dash="dash"),                                               ### PLOTTING
)                                                                                                   ### PLOTTING
stationary_example_mc_figure.add_hline(                                                             ### PLOTTING
    y=stationary_example_h_speed_index,                                                             ### PLOTTING
    line=dict(color="#7f7f7f", width=2, dash="dash"),                                               ### PLOTTING
)                                                                                                   ### PLOTTING
stationary_example_mc_figure.add_annotation(                                                        ### PLOTTING
    x=1.0,                                                                                          ### PLOTTING
    y=stationary_example_mc_oracle_paper_y,                                                         ### PLOTTING
    xref="paper",                                                                                   ### PLOTTING
    yref="paper",                                                                                   ### PLOTTING
    text=(                                                                                          ### PLOTTING
        "Gaussian MISE oracle: "                                                                    ### PLOTTING
        f"index={stationary_example_mc_oracle_index:.2f}, "                                         ### PLOTTING
        f"h={stationary_example_oracle_h:.4g}"                                                      ### PLOTTING
    ),                                                                                              ### PLOTTING
    showarrow=False,                                                                                ### PLOTTING
    xanchor="right",                                                                                ### PLOTTING
    yanchor="bottom",                                                                               ### PLOTTING
    font=dict(color="#d62728"),                                                                     ### PLOTTING
)                                                                                                   ### PLOTTING
stationary_example_mc_figure.add_annotation(                                                        ### PLOTTING
    x=0.0,                                                                                          ### PLOTTING
    y=stationary_example_mc_h_speed_paper_y,                                                        ### PLOTTING
    xref="paper",                                                                                   ### PLOTTING
    yref="paper",                                                                                   ### PLOTTING
    text=(                                                                                          ### PLOTTING
        "theoretical speed bandwidth: "                                                             ### PLOTTING
        f"index={stationary_example_h_speed_index:.2f}, "                                           ### PLOTTING
        f"h={stationary_example_h_speed:.4g}"                                                       ### PLOTTING
    ),                                                                                              ### PLOTTING
    showarrow=False,                                                                                ### PLOTTING
    xanchor="left",                                                                                 ### PLOTTING
    yanchor="bottom",                                                                               ### PLOTTING
    font=dict(color="#7f7f7f"),                                                                     ### PLOTTING
    bgcolor="rgba(255,255,255,0.8)",                                                                ### PLOTTING
)                                                                                                   ### PLOTTING
stationary_example_mc_figure.update_xaxes(                                                          ### PLOTTING
    title_text="similarity scale λ",                                                                ### PLOTTING
    type="log",                                                                                     ### PLOTTING
    range=[                                                                                         ### PLOTTING
        stationary_example_mc_log_lambda_min,                                                       ### PLOTTING
        stationary_example_mc_log_lambda_max,                                                       ### PLOTTING
    ],                                                                                              ### PLOTTING
)                                                                                                   ### PLOTTING
stationary_example_mc_figure.update_yaxes(                                                          ### PLOTTING
    title_text="chosen bandwidth grid index",                                                       ### PLOTTING
    range=[-0.5, stationary_example_N - 0.5],                                                       ### PLOTTING
    tickmode="array",                                                                               ### PLOTTING
    tickvals=stationary_example_mc_tick_indices,                                                    ### PLOTTING
    ticktext=stationary_example_mc_tick_labels,                                                     ### PLOTTING
)                                                                                                   ### PLOTTING
stationary_example_mc_figure.update_layout(                                                         ### PLOTTING
    width=1050,                                                                                     ### PLOTTING
    height=600,                                                                                     ### PLOTTING
    template="plotly_white",                                                                        ### PLOTTING
    legend=dict(                                                                                    ### PLOTTING
        orientation="h",                                                                            ### PLOTTING
        x=0.5,                                                                                      ### PLOTTING
        xanchor="center",                                                                           ### PLOTTING
        y=-0.18,                                                                                    ### PLOTTING
        yanchor="top",                                                                              ### PLOTTING
    ),                                                                                              ### PLOTTING
    margin=dict(t=100, b=150),                                                                      ### PLOTTING
)                                                                                                   ### PLOTTING
save_thesis_plotly_figure(                                                                          ### PLOTTING
    stationary_example_mc_figure,                                                                   ### PLOTTING
    "chosen_bandwidth_vs_lambda.png",                                                               ### PLOTTING
)                                                                                                   ### PLOTTING
stationary_example_mc_figure.show()                                                                 ### PLOTTING


### Grid-fineness sensitivity across similarity scales

This experiment reuses the same fixed sample and bandwidth interval while varying the number $N=11,16,\ldots,1001$ of bandwidth candidates linear in $\psi_\beta$. It plots $N$ against the bandwidth selected by the Perron-vector maximum for $\lambda\in\{1,10,100,1000\}$. The curve colours match Figure 3.1. Envelope fits are omitted so that the four grid-stability trajectories remain directly comparable. A thin grey vertical line at $N=201$ identifies the grid size used in Figure 3.3.


In [ ]:
stationary_example_grid_fineness_lambdas = (
    stationary_example_lambdas.copy()
)
stationary_example_grid_fineness_N_values = np.arange(
    11, 1002, 5, dtype=int
)
stationary_example_grid_fineness_tick_values = np.arange(
    101, 1002, 100, dtype=int
)
stationary_example_grid_fineness_selected_indices = np.empty(
    (
        len(stationary_example_grid_fineness_lambdas),
        len(stationary_example_grid_fineness_N_values),
    ),
    dtype=int,
)
stationary_example_grid_fineness_selected_bandwidths = np.empty(
    stationary_example_grid_fineness_selected_indices.shape,
    dtype=float,
)

for position, current_N in enumerate(
    stationary_example_grid_fineness_N_values
):
    current_grid = bandwidth_grid_linear_in_psi_beta(
        h_min=stationary_example_h_min,
        h_max=stationary_example_h_max,
        beta=stationary_example_beta,
        grid_size=int(current_N),
    )
    current_divergence = beta2_divergence_matrix_fft(
        samples=stationary_example_samples,
        h_grid=current_grid,
        half_width=stationary_example_half_width,
        spatial_points=stationary_example_spatial_points,
    )

    for lambda_position, current_lambda in enumerate(
        stationary_example_grid_fineness_lambdas
    ):
        current_perron = perron_vector_from_divergence(
            current_divergence, current_lambda
        )
        current_stationary_mass = doob_mass_from_perron(current_perron)
        current_selected_index = int(
            perron_peak_index(current_perron)
        )
        stationary_example_grid_fineness_selected_indices[
            lambda_position, position
        ] = current_selected_index
        stationary_example_grid_fineness_selected_bandwidths[
            lambda_position, position
        ] = current_grid[current_selected_index]

    if (position + 1) % 20 == 0 or position + 1 == len(                                             ### REPORTING
        stationary_example_grid_fineness_N_values                                                   ### REPORTING
    ):                                                                                              ### REPORTING
        print(                                                                                      ### REPORTING
            f"completed {position + 1}/"                                                            ### REPORTING
            f"{len(stationary_example_grid_fineness_N_values)} grid sizes"                          ### REPORTING
        )                                                                                           ### REPORTING

fig, ax = plt.subplots(figsize=(10.5, 5.5))                                                         ### PLOTTING

for lambda_position, (current_lambda, current_color) in enumerate(                                  ### PLOTTING
    zip(                                                                                            ### PLOTTING
        stationary_example_grid_fineness_lambdas,                                                   ### PLOTTING
        stationary_example_curve_colors,                                                            ### PLOTTING
    )                                                                                               ### PLOTTING
):                                                                                                  ### PLOTTING
    ax.plot(                                                                                        ### PLOTTING
        stationary_example_grid_fineness_N_values,                                                  ### PLOTTING
        stationary_example_grid_fineness_selected_bandwidths[                                       ### PLOTTING
            lambda_position                                                                         ### PLOTTING
        ],                                                                                          ### PLOTTING
        color=current_color,                                                                        ### PLOTTING
        linewidth=1.8,                                                                              ### PLOTTING
        marker="o",                                                                                 ### PLOTTING
        markersize=2.8,                                                                             ### PLOTTING
        label=rf"$\lambda={current_lambda:g}$",                                                     ### PLOTTING
    )                                                                                               ### PLOTTING

ax.set_xlabel("number of bandwidth candidates N")                                                   ### PLOTTING
ax.set_ylabel("chosen bandwidth h")                                                                 ### PLOTTING
ax.set_xticks(stationary_example_grid_fineness_tick_values)                                         ### PLOTTING
ax.grid(alpha=0.3)                                                                                  ### PLOTTING
ax.axvline(                                                                                         ### PLOTTING
    stationary_example_lambda_grid_N,                                                               ### PLOTTING
    color="#7f7f7f",                                                                                ### PLOTTING
    linewidth=1.0,                                                                                  ### PLOTTING
    label=rf"Figure 3.3 uses $N={stationary_example_lambda_grid_N}$",                               ### PLOTTING
    zorder=0,                                                                                       ### PLOTTING
)                                                                                                   ### PLOTTING
ax.legend(ncol=3, fontsize=8.5)                                                                     ### PLOTTING
plt.tight_layout()                                                                                  ### PLOTTING
fig.savefig(                                                                                        ### PLOTTING
    kde_thesis_figure_dir / "grid_fineness_chosen_bandwidths.pdf",                                  ### PLOTTING
    dpi=100,                                                                                        ### PLOTTING
)                                                                                                   ### PLOTTING
plt.show()                                                                                          ### PLOTTING


## 2. Configurable beta-2 selection histogram

Choose $n$, $N$, $M$, the bandwidth interval, and the thesis similarity parameter $\lambda=1/\tau$ below. Each repetition selects the largest component of the Doob stationary distribution on the fixed geometric bandwidth grid.

Figure 3.5 uses `beta2_N = 1000` and `beta2_binning = 5`. The plotting cell saves the updated PDF directly to `outputs/images/thesis/distribution_of_chosen_bandwidths.pdf`, the file included by the thesis.


In [ ]:
beta2_n = 200
beta2_N = 1000
beta2_M = 1000
beta2_binning = 5  # Consecutive bandwidth-grid positions per histogram bin.

beta2_h_min = 0.01
beta2_h_max = 2.0
beta2_lambda = 1.0
beta2_seed = seed + 10
beta2_half_width = 12.0
beta2_spatial_points = 8192
beta2_validate_fft = True

if beta2_n < 1 or beta2_M < 1:
    raise ValueError("beta2_n and beta2_M must be positive.")
if beta2_binning < 1 or int(beta2_binning) != beta2_binning:
    raise ValueError("beta2_binning must be a positive integer.")
if beta2_N < 2:
    raise ValueError("beta2_N must be at least 2.")
if not 0.0 < beta2_h_min < beta2_h_max:
    raise ValueError("Require 0 < beta2_h_min < beta2_h_max.")
if beta2_lambda <= 0.0:
    raise ValueError("beta2_lambda must be positive.")
if beta2_half_width <= 0.0:
    raise ValueError("beta2_half_width must be positive.")
if beta2_spatial_points < 2:
    raise ValueError("beta2_spatial_points must be at least 2.")

beta2_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=beta2_h_min,
    h_max=beta2_h_max,
    beta=2.0,
    grid_size=beta2_N,
)

print(                                                                                              ### REPORTING
    f"beta=2, n={beta2_n}, N={beta2_N}, M={beta2_M}, "                                              ### REPORTING
    f"lambda={beta2_lambda:g}"                                                                      ### REPORTING
)                                                                                                   ### REPORTING

In [ ]:
if beta2_validate_fft:
    validation_rng = np.random.default_rng(beta2_seed)
    validation_samples = validation_rng.normal(
        size=min(beta2_n, 100)
    )
    validation_indices = np.unique(
        np.linspace(0, beta2_N - 1, min(beta2_N, 20), dtype=int)
    )
    validation_h_grid = beta2_h_grid[validation_indices]
    validation_exact = beta2_divergence_matrix(
        samples=validation_samples,
        h_grid=validation_h_grid,
    )
    validation_fft = beta2_divergence_matrix_fft(
        samples=validation_samples,
        h_grid=validation_h_grid,
        half_width=beta2_half_width,
        spatial_points=beta2_spatial_points,
    )
    beta2_fft_maximum_absolute_error = float(
        np.max(np.abs(validation_fft - validation_exact))
    )
    print(                                                                                          ### REPORTING
        "FFT validation maximum absolute error: "                                                   ### REPORTING
        f"{beta2_fft_maximum_absolute_error:.3e}"                                                   ### REPORTING
    )                                                                                               ### REPORTING

def exact_beta2_qhat_values(
    samples: np.ndarray,
    h_grid: np.ndarray,
) -> np.ndarray:
    """Evaluate Q-hat_(n,2) exactly on a bandwidth grid."""
    squared_differences = (
        samples[:, None] - samples[None, :]
    ) ** 2
    values = np.empty(len(h_grid), dtype=float)
    for index, bandwidth in enumerate(h_grid):
        q = 2.0 * bandwidth**2
        overlap = (
            np.exp(-squared_differences / (2.0 * q))
            / np.sqrt(2.0 * np.pi * q)
        )
        second_q_factor = (
            3.0 / (4.0 * q**2)
            - 3.0 * squared_differences / (2.0 * q**3)
            + squared_differences**2 / (4.0 * q**4)
        )
        values[index] = (
            2.0
            * bandwidth**4
            * np.mean(overlap * second_q_factor)
        )
    return values

beta2_rng = np.random.default_rng(beta2_seed)
beta2_selected_indices = np.empty(beta2_M, dtype=int)
beta2_ise_selected_indices = np.empty(beta2_M, dtype=int)
beta2_qhat_selected_indices = np.empty(beta2_M, dtype=int)

for repetition in range(beta2_M):
    current_samples = beta2_rng.normal(size=beta2_n)
    current_divergence = beta2_divergence_matrix_fft(
        samples=current_samples,
        h_grid=beta2_h_grid,
        half_width=beta2_half_width,
        spatial_points=beta2_spatial_points,
    )
    current_perron = perron_vector_from_divergence(current_divergence, beta2_lambda)
    current_stationary = doob_mass_from_perron(current_perron)
    beta2_selected_indices[repetition] = int(
        perron_peak_index(current_perron)
    )
    current_ise_values = exact_oracle_ise_values(
        samples=current_samples,
        h_grid=beta2_h_grid,
        sigma2=1.0,
    )
    beta2_ise_selected_indices[repetition] = int(
        np.argmin(current_ise_values)
    )
    current_qhat_values = exact_beta2_qhat_values(
        samples=current_samples,
        h_grid=beta2_h_grid,
    )
    beta2_qhat_selected_indices[repetition] = int(
        np.argmin(current_qhat_values)
    )

    if (repetition + 1) % 5 == 0 or repetition + 1 == beta2_M:                                      ### REPORTING
        print(f"completed {repetition + 1}/{beta2_M} repetitions")                                  ### REPORTING

beta2_selection_counts = np.bincount(
    beta2_selected_indices,
    minlength=beta2_N,
)
beta2_ise_selection_counts = np.bincount(
    beta2_ise_selected_indices,
    minlength=beta2_N,
)
beta2_qhat_selection_counts = np.bincount(
    beta2_qhat_selected_indices,
    minlength=beta2_N,
)


In [ ]:
from matplotlib.ticker import MultipleLocator, PercentFormatter

# Figure 3.5: select on the full N-point grid and group only the display bins.
beta2_histogram_counts = np.stack([
    beta2_selection_counts,
    beta2_ise_selection_counts,
    beta2_qhat_selection_counts,
])
assert beta2_histogram_counts.shape == (3, beta2_N)
beta2_bin_starts = np.arange(0, beta2_N, beta2_binning)
beta2_bin_stops = np.minimum(beta2_bin_starts + beta2_binning, beta2_N)
beta2_bin_centers = (beta2_bin_starts + beta2_bin_stops - 1) / 2.0
beta2_bin_widths = beta2_bin_stops - beta2_bin_starts
beta2_binned_counts = np.add.reduceat(beta2_histogram_counts, beta2_bin_starts, axis=1)
assert np.array_equal(beta2_binned_counts.sum(axis=1), beta2_histogram_counts.sum(axis=1))
assert np.all(beta2_binned_counts.sum(axis=1) == beta2_M)

beta2_distribution_names = [
    "spectral selections", "ISE-optimal grid bandwidths", "Q-hat minimizers",
]
beta2_histogram, beta2_axis = plt.subplots(figsize=(12.0, 6.32))
beta2_histogram.subplots_adjust(left=0.085, right=0.98, top=0.88, bottom=0.25)
for beta2_label, beta2_color, beta2_opacity, beta2_values in zip(
    beta2_distribution_names,
    ["#C65D00", "#9467BD", "#66C2A5"],
    [0.70, 0.70, 0.55],
    beta2_binned_counts,
):
    beta2_nonzero = np.flatnonzero(beta2_values)
    beta2_axis.bar(
        beta2_bin_centers[beta2_nonzero], beta2_values[beta2_nonzero] / beta2_M,
        width=0.94 * beta2_bin_widths[beta2_nonzero], color=beta2_color,
        alpha=beta2_opacity, edgecolor="white", linewidth=0.3, label=beta2_label,
    )

beta2_mise_bandwidth = gaussian_mise_oracle_bandwidth(beta2_n)
beta2_grid_indices = np.arange(beta2_N, dtype=float)
beta2_mise_grid_position = float(np.interp(
    np.log(beta2_mise_bandwidth), np.log(beta2_h_grid), beta2_grid_indices,
))
beta2_reference_label = f"Gaussian MISE optimum: h={beta2_mise_bandwidth:.4g}"
beta2_axis.axvline(
    beta2_mise_grid_position, color="#1f77b4", linestyle=(0, (5, 5)),
    linewidth=1.5, label=beta2_reference_label,
)
beta2_ticks = np.unique(np.linspace(0, beta2_N - 1, 7, dtype=int))
beta2_axis.set_xticks(beta2_ticks, [f"{beta2_h_grid[i]:.3g}" for i in beta2_ticks])
beta2_axis.set_xlim(-0.5, beta2_N - 0.5)
beta2_axis.set_ylim(0, 1.05 * float(beta2_binned_counts.max()) / beta2_M)
beta2_axis.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
beta2_axis.yaxis.set_major_locator(MultipleLocator(0.01))
beta2_axis.set_xlabel("selected bandwidth h", fontsize=12, labelpad=12, color="#304967")
beta2_axis.set_ylabel("relative frequency", fontsize=12, labelpad=12, color="#304967")
beta2_axis.tick_params(axis="both", labelsize=10, colors="#304967", length=0)
beta2_axis.grid(axis="y", color="#E5ECF6", linewidth=0.6)
beta2_axis.set_axisbelow(True)
for beta2_side in ("top", "right", "left"):
    beta2_axis.spines[beta2_side].set_visible(False)
beta2_axis.spines["bottom"].set_color("#E5ECF6")
beta2_histogram.suptitle(
    f"Figure 3.5: {beta2_binning} consecutive grid points per bin\n"
    + rf"$N={beta2_N}$, $n={beta2_n}$, $M={beta2_M}$, $\lambda={beta2_lambda:g}$",
    y=0.965, fontsize=14, color="#304967",
)
beta2_handles, beta2_legend_labels = beta2_axis.get_legend_handles_labels()
beta2_legend_order = [
    beta2_legend_labels.index(label)
    for label in [*beta2_distribution_names, beta2_reference_label]
]
beta2_histogram.legend(
    [beta2_handles[i] for i in beta2_legend_order],
    [beta2_legend_labels[i] for i in beta2_legend_order],
    loc="lower center", bbox_to_anchor=(0.52, 0.035), ncol=2, frameon=False,
    fontsize=10, labelcolor="#304967",
)

# Replace the tracked PDF already included by the thesis and Overleaf.
beta2_thesis_figure_path = kde_thesis_figure_dir / "distribution_of_chosen_bandwidths.pdf"
beta2_histogram.savefig(beta2_thesis_figure_path, format="pdf", bbox_inches="tight")
print(f"Updated Figure 3.5: {beta2_thesis_figure_path}")
print(f"N={beta2_N}; binning={beta2_binning}; bins={len(beta2_bin_starts)}")
plt.show()  # The shared exporter also saves a PDF under outputs/pdfs/spectral_scale_selection_kde/.
plt.close(beta2_histogram)


## 3. Beta-2 histograms for increasing sample size

This repeats the FFT-accelerated selection for $n=100,200,\ldots,1000$ on one fixed geometric grid. Prefixes of the same maximum-size sample are reused within each repetition. The plotting cell produces one standalone histogram per sample size, one colored overlay of all histograms, and one colored overlay controlled by the editable list `beta2_n_selected_overlay_values`. Dashed vertical lines in the standalone figures mark the nearest grid indices to the geometric center and the Gaussian MISE-oracle bandwidth.


In [ ]:
beta2_n_sweep_values = np.arange(100, 1001, 100, dtype=int)
beta2_n_sweep_N = 201
beta2_n_sweep_M = 1000
beta2_n_sweep_h_min = beta2_h_min
beta2_n_sweep_h_max = beta2_h_max
beta2_n_sweep_lambda = beta2_lambda
beta2_n_sweep_seed = beta2_seed + 2

beta2_n_sweep_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=beta2_n_sweep_h_min,
    h_max=beta2_n_sweep_h_max,
    beta=2.0,
    grid_size=beta2_n_sweep_N,
)
beta2_n_sweep_grid_indices = np.arange(
    beta2_n_sweep_N, dtype=float
)
beta2_n_sweep_oracle_bandwidths = {
    int(current_n): gaussian_mise_oracle_bandwidth(int(current_n))
    for current_n in beta2_n_sweep_values
}
beta2_n_sweep_oracle_indices = {
    int(current_n): float(np.interp(
        np.log(beta2_n_sweep_oracle_bandwidths[int(current_n)]),
        np.log(beta2_n_sweep_h_grid),
        beta2_n_sweep_grid_indices,
    ))
    for current_n in beta2_n_sweep_values
}
beta2_n_sweep_dx = (
    2.0 * beta2_half_width / beta2_spatial_points
)
beta2_n_sweep_spatial_grid = (
    -beta2_half_width
    + beta2_n_sweep_dx * np.arange(beta2_spatial_points)
)
beta2_n_sweep_true_density = (
    np.exp(-0.5 * beta2_n_sweep_spatial_grid**2)
    / np.sqrt(2.0 * np.pi)
)
beta2_n_sweep_frequencies = 2.0 * np.pi * np.fft.rfftfreq(
    beta2_spatial_points, d=beta2_n_sweep_dx
)
beta2_n_sweep_parseval_weights = np.full(
    len(beta2_n_sweep_frequencies), 2.0
)
beta2_n_sweep_parseval_weights[0] = 1.0
beta2_n_sweep_parseval_weights[-1] = 1.0

beta2_n_sweep_rng = np.random.default_rng(beta2_n_sweep_seed)
beta2_n_sweep_full_samples = beta2_n_sweep_rng.normal(
    size=(beta2_n_sweep_M, int(np.max(beta2_n_sweep_values)))
)
beta2_n_sweep_results = {}

for current_n in beta2_n_sweep_values:
    selected_indices = np.empty(beta2_n_sweep_M, dtype=int)
    ise_selected_indices = np.empty(beta2_n_sweep_M, dtype=int)
    qhat_selected_indices = np.empty(beta2_n_sweep_M, dtype=int)

    for repetition in range(beta2_n_sweep_M):
        current_samples = beta2_n_sweep_full_samples[
            repetition, : int(current_n)
        ]
        current_densities, current_dx = kde_family_fft(
            samples=current_samples,
            bandwidths=beta2_n_sweep_h_grid,
            half_width=beta2_half_width,
            spatial_points=beta2_spatial_points,
        )
        current_gram = current_dx * (
            current_densities @ current_densities.T
        )
        current_norms = np.diag(current_gram)
        current_squared_distances = (
            current_norms[:, None]
            + current_norms[None, :]
            - 2.0 * current_gram
        )
        np.maximum(
            current_squared_distances,
            0.0,
            out=current_squared_distances,
        )
        current_perron = perron_vector_from_divergence(
            0.5 * current_squared_distances, beta2_n_sweep_lambda
        )
        selected_indices[repetition] = int(perron_peak_index(current_perron))
        current_ise_values = current_dx * np.sum(
            (
                current_densities
                - beta2_n_sweep_true_density[None, :]
            ) ** 2,
            axis=1,
        )
        ise_selected_indices[repetition] = int(
            np.argmin(current_ise_values)
        )
        current_density_coefficients = np.fft.rfft(
            current_densities, axis=1
        )
        current_log_derivative_coefficients = (
            -beta2_n_sweep_h_grid[:, None] ** 2
            * beta2_n_sweep_frequencies[None, :] ** 2
            * current_density_coefficients
        )
        current_qhat_values = (
            0.5
            * current_dx
            / beta2_spatial_points
            * np.sum(
                beta2_n_sweep_parseval_weights[None, :]
                * np.abs(
                    current_log_derivative_coefficients
                ) ** 2,
                axis=1,
            )
        )
        qhat_selected_indices[repetition] = int(
            np.argmin(current_qhat_values)
        )

    selection_counts = np.bincount(
        selected_indices,
        minlength=beta2_n_sweep_N,
    )
    ise_selection_counts = np.bincount(
        ise_selected_indices,
        minlength=beta2_n_sweep_N,
    )
    qhat_selection_counts = np.bincount(
        qhat_selected_indices,
        minlength=beta2_n_sweep_N,
    )
    beta2_n_sweep_results[int(current_n)] = {
        "selected_indices": selected_indices.copy(),
        "counts": selection_counts,
        "relative_frequencies": selection_counts / beta2_n_sweep_M,
        "ise_selected_indices": ise_selected_indices.copy(),
        "ise_counts": ise_selection_counts,
        "qhat_selected_indices": qhat_selected_indices.copy(),
        "qhat_counts": qhat_selection_counts,
        "oracle_bandwidth": beta2_n_sweep_oracle_bandwidths[
            int(current_n)
        ],
        "oracle_index": beta2_n_sweep_oracle_indices[int(current_n)],
    }
    print(f"completed n={current_n}")                                                               ### REPORTING


In [ ]:
beta2_n_selected_overlay_values = [100, 500, 1000]

beta2_n_available_values = {
    int(value) for value in beta2_n_sweep_values
}
beta2_n_selected_overlay_values = [
    int(value) for value in beta2_n_selected_overlay_values
]
beta2_n_invalid_selected_values = sorted(                                                           ### PLOTTING
    set(beta2_n_selected_overlay_values)                                                            ### PLOTTING
    - beta2_n_available_values                                                                      ### PLOTTING
)                                                                                                   ### PLOTTING
if beta2_n_invalid_selected_values:
    raise ValueError(
        "Selected overlay sample sizes are unavailable: "
        f"{beta2_n_invalid_selected_values}. Choose from "
        f"{sorted(beta2_n_available_values)}."
    )
if not beta2_n_selected_overlay_values:
    raise ValueError(
        "beta2_n_selected_overlay_values must not be empty."
    )

beta2_n_individual_figures = {}

for current_n in beta2_n_sweep_values:
    current_n = int(current_n)
    result = beta2_n_sweep_results[current_n]
    current_figure = plot_bandwidth_distributions(                                                  ### PLOTTING
        h_grid=beta2_n_sweep_h_grid,                                                                ### PLOTTING
        distributions={                                                                             ### PLOTTING
            "spectral selections": result["counts"],                                                ### PLOTTING
            "ISE-optimal grid bandwidths": result["ise_counts"],                                    ### PLOTTING
            "Q-hat minimizers": result["qhat_counts"],                                              ### PLOTTING
        },                                                                                          ### PLOTTING
        reference_lines=[(                                                                          ### PLOTTING
            (                                                                                       ### PLOTTING
                "Gaussian MISE optimum: "                                                           ### PLOTTING
                f"h={result['oracle_bandwidth']:.4g}"                                               ### PLOTTING
            ),                                                                                      ### PLOTTING
            result["oracle_index"],                                                                 ### PLOTTING
            "#1f77b4",                                                                              ### PLOTTING
        )],                                                                                         ### PLOTTING
        colors={                                                                                    ### PLOTTING
            "spectral selections": "#C65D00",                                                       ### PLOTTING
            "ISE-optimal grid bandwidths": "#9467BD",                                               ### PLOTTING
            "Q-hat minimizers": "#66C2A5",                                                          ### PLOTTING
        },                                                                                          ### PLOTTING
        opacity=0.70,                                                                               ### PLOTTING
    )                                                                                               ### PLOTTING
    for trace in current_figure.data:
        if trace.name == "Q-hat minimizers":
            trace.opacity = 0.55
    beta2_n_individual_figures[current_n] = current_figure
    if current_n in {300, 400, 500, 1000}:                                                          ### PLOTTING
        save_thesis_plotly_figure(                                                                  ### PLOTTING
            current_figure,                                                                         ### PLOTTING
            f"distribution_of_chosen_bandwidths_n{current_n}.pdf",                                  ### PLOTTING
        )                                                                                           ### PLOTTING
    current_figure.show()                                                                           ### PLOTTING

beta2_n_overlay = plot_bandwidth_distributions(                                                     ### PLOTTING
    h_grid=beta2_n_sweep_h_grid,                                                                    ### PLOTTING
    distributions={                                                                                 ### PLOTTING
        f"n={int(current_n)}": beta2_n_sweep_results[                                               ### PLOTTING
            int(current_n)                                                                          ### PLOTTING
        ]["counts"]                                                                                 ### PLOTTING
        for current_n in beta2_n_sweep_values                                                       ### PLOTTING
    },                                                                                              ### PLOTTING
    opacity=0.24,                                                                                   ### PLOTTING
    width=1100,                                                                                     ### PLOTTING
    height=650,                                                                                     ### PLOTTING
    legend_title="sample size",                                                                     ### PLOTTING
)                                                                                                   ### PLOTTING
beta2_n_overlay.show()                                                                              ### PLOTTING

beta2_n_selected_overlay = plot_bandwidth_distributions(                                            ### PLOTTING
    h_grid=beta2_n_sweep_h_grid,                                                                    ### PLOTTING
    distributions={                                                                                 ### PLOTTING
        f"n={current_n}": beta2_n_sweep_results[current_n][                                         ### PLOTTING
            "counts"                                                                                ### PLOTTING
        ]                                                                                           ### PLOTTING
        for current_n in beta2_n_selected_overlay_values                                            ### PLOTTING
    },                                                                                              ### PLOTTING
    opacity=0.32,                                                                                   ### PLOTTING
    width=1100,                                                                                     ### PLOTTING
    height=650,                                                                                     ### PLOTTING
    legend_title="sample size",                                                                     ### PLOTTING
)                                                                                                   ### PLOTTING
beta2_n_selected_overlay.show()                                                                     ### PLOTTING


### Median bandwidth path and theoretical-rate comparison

The histograms above already contain repeated selections for every sample size. Without running another simulation, this block summarizes those same selections by their median and interquartile range and compares them with

$$
h_{\mathrm{AMISE}}(n)=\left(\frac{4}{3n}\right)^{1/5},
\qquad
h_{\mathrm{speed}}(n)=\left(\frac{1}{4n}\right)^{1/5},
$$

and the exact finite-sample Gaussian MISE oracle. The second panel reports the selected bandwidth relative to $h_{\mathrm{speed}}(n)$. Here the bandwidth grid, its size $N$, and the similarity scale $\lambda$ remain fixed while only the sample size increases.


In [ ]:
beta2_n_sweep_selected_bandwidth_samples = np.vstack([
    beta2_n_sweep_h_grid[
        beta2_n_sweep_results[int(current_n)]["selected_indices"]
    ]
    for current_n in beta2_n_sweep_values
])
(
    beta2_n_sweep_q25_bandwidths,
    beta2_n_sweep_median_bandwidths,
    beta2_n_sweep_q75_bandwidths,
) = np.quantile(
    beta2_n_sweep_selected_bandwidth_samples,
    [0.25, 0.50, 0.75],
    axis=1,
)

beta2_n_sweep_amise_bandwidths = (
    4.0 / (3.0 * beta2_n_sweep_values)
) ** (1.0 / 5.0)
beta2_n_sweep_speed_bandwidths = (
    1.0 / (4.0 * beta2_n_sweep_values)
) ** (1.0 / 5.0)
beta2_n_sweep_exact_oracle_bandwidths = np.array([
    beta2_n_sweep_oracle_bandwidths[int(current_n)]
    for current_n in beta2_n_sweep_values
])

beta2_n_sweep_speed_ratio_samples = (
    beta2_n_sweep_selected_bandwidth_samples
    / beta2_n_sweep_speed_bandwidths[:, None]
)
(
    beta2_n_sweep_q25_speed_ratios,
    beta2_n_sweep_median_speed_ratios,
    beta2_n_sweep_q75_speed_ratios,
) = np.quantile(
    beta2_n_sweep_speed_ratio_samples,
    [0.25, 0.50, 0.75],
    axis=1,
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))                                                   ### PLOTTING

axes[0].fill_between(                                                                               ### PLOTTING
    beta2_n_sweep_values,                                                                           ### PLOTTING
    beta2_n_sweep_q25_bandwidths,                                                                   ### PLOTTING
    beta2_n_sweep_q75_bandwidths,                                                                   ### PLOTTING
    color="tab:green",                                                                              ### PLOTTING
    alpha=0.18,                                                                                     ### PLOTTING
    label="spectral interquartile range",                                                           ### PLOTTING
)                                                                                                   ### PLOTTING
axes[0].plot(                                                                                       ### PLOTTING
    beta2_n_sweep_values,                                                                           ### PLOTTING
    beta2_n_sweep_median_bandwidths,                                                                ### PLOTTING
    color="tab:green",                                                                              ### PLOTTING
    marker="o",                                                                                     ### PLOTTING
    linewidth=2.2,                                                                                  ### PLOTTING
    label=(                                                                                         ### PLOTTING
        rf"median spectral $\widehat{{h}}_n$ "                                                      ### PLOTTING
        rf"($\lambda={beta2_n_sweep_lambda:g}$)"                                                    ### PLOTTING
    ),                                                                                              ### PLOTTING
)                                                                                                   ### PLOTTING
axes[0].plot(                                                                                       ### PLOTTING
    beta2_n_sweep_values,                                                                           ### PLOTTING
    beta2_n_sweep_amise_bandwidths,                                                                 ### PLOTTING
    linewidth=2.0,                                                                                  ### PLOTTING
    label=r"$h_{\mathrm{AMISE}}(n)$",                                                               ### PLOTTING
)                                                                                                   ### PLOTTING
axes[0].plot(                                                                                       ### PLOTTING
    beta2_n_sweep_values,                                                                           ### PLOTTING
    beta2_n_sweep_speed_bandwidths,                                                                 ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    linewidth=2.0,                                                                                  ### PLOTTING
    label=r"$h_{\mathrm{speed}}(n)$",                                                               ### PLOTTING
)                                                                                                   ### PLOTTING
axes[0].plot(                                                                                       ### PLOTTING
    beta2_n_sweep_values,                                                                           ### PLOTTING
    beta2_n_sweep_exact_oracle_bandwidths,                                                          ### PLOTTING
    linestyle="-.",                                                                                 ### PLOTTING
    linewidth=2.0,                                                                                  ### PLOTTING
    label="exact Gaussian MISE oracle",                                                             ### PLOTTING
)                                                                                                   ### PLOTTING
axes[0].set_xlabel("sample size n")                                                                 ### PLOTTING
axes[0].set_ylabel("bandwidth h")                                                                   ### PLOTTING
axes[0].set_yscale("log")                                                                           ### PLOTTING
axes[0].set_title("Selected bandwidth and theoretical references")
axes[0].legend()                                                                                    ### PLOTTING

axes[1].fill_between(                                                                               ### PLOTTING
    beta2_n_sweep_values,                                                                           ### PLOTTING
    beta2_n_sweep_q25_speed_ratios,                                                                 ### PLOTTING
    beta2_n_sweep_q75_speed_ratios,                                                                 ### PLOTTING
    color="tab:purple",                                                                             ### PLOTTING
    alpha=0.18,                                                                                     ### PLOTTING
    label="interquartile range",                                                                    ### PLOTTING
)                                                                                                   ### PLOTTING
axes[1].plot(                                                                                       ### PLOTTING
    beta2_n_sweep_values,                                                                           ### PLOTTING
    beta2_n_sweep_median_speed_ratios,                                                              ### PLOTTING
    color="tab:purple",                                                                             ### PLOTTING
    marker="o",                                                                                     ### PLOTTING
    linewidth=2.2,                                                                                  ### PLOTTING
    label=r"median $\widehat h_n/h_{\mathrm{speed}}(n)$",                                           ### PLOTTING
)                                                                                                   ### PLOTTING
axes[1].axhline(                                                                                    ### PLOTTING
    1.0,                                                                                            ### PLOTTING
    color="black",                                                                                  ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    linewidth=1.5,                                                                                  ### PLOTTING
    label="target ratio = 1",                                                                       ### PLOTTING
)                                                                                                   ### PLOTTING
axes[1].set_xlabel("sample size n")                                                                 ### PLOTTING
axes[1].set_ylabel(r"$\widehat h_n/h_{\mathrm{speed}}(n)$")                                         ### PLOTTING
axes[1].set_title("Spectral bandwidth relative to theoretical speed")
axes[1].legend()                                                                                    ### PLOTTING

for ax in axes:                                                                                     ### PLOTTING
    ax.grid(alpha=0.3)                                                                              ### PLOTTING

plt.tight_layout()                                                                                  ### PLOTTING
plt.show()                                                                                          ### PLOTTING


## 4. Sample, exact-population, and first-order Perron vectors

This experiment compares three sum-normalized Perron vectors on the same bandwidth grid:

1. the sample vector obtained from the FFT beta-2 similarity matrix;
2. the exact population vector obtained by assembling the analytic Gaussian population matrix and solving its Perron eigenproblem numerically; and
3. the first-order perturbation formula

$$
v_i^{(1)}
=\frac{1}{N}
-\frac{\lambda}{N^2}\left(r_i-\bar r\right),
\qquad
r_i=\sum_j D_{ij}^{\mathrm{pop}}.
$$

The implicit row-sum characterization determines the maximum of the first-order vector; it does not by itself provide the full exact Perron vector. All curves are displayed as $Nv_i$, so the horizontal value one corresponds to the uniform vector.

In [ ]:
perron_comparison_n = 1000
perron_comparison_N = 191
perron_comparison_h_min = 0.01
perron_comparison_h_max = 2.0
perron_comparison_lambda = 50.0
# Match the first repetition of the beta-2 sample-size sweep.
perron_comparison_seed = seed + 12
perron_comparison_half_width = beta2_half_width
perron_comparison_spatial_points = beta2_spatial_points

if perron_comparison_n < 1 or perron_comparison_N < 2:
    raise ValueError("Require n >= 1 and N >= 2.")
if not 0.0 < perron_comparison_h_min < perron_comparison_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if perron_comparison_lambda < 0.0:
    raise ValueError("perron_comparison_lambda must be nonnegative.")

perron_comparison_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=perron_comparison_h_min,
    h_max=perron_comparison_h_max,
    beta=2.0,
    grid_size=perron_comparison_N,
)
perron_comparison_rng = np.random.default_rng(perron_comparison_seed)
perron_comparison_samples = perron_comparison_rng.normal(
    size=perron_comparison_n
)

perron_comparison_sample_divergence = beta2_divergence_matrix_fft(
    samples=perron_comparison_samples,
    h_grid=perron_comparison_h_grid,
    half_width=perron_comparison_half_width,
    spatial_points=perron_comparison_spatial_points,
)
perron_comparison_sample_vector = perron_vector_from_divergence(
    perron_comparison_sample_divergence, perron_comparison_lambda
)

perron_comparison_population_divergence = (
    population_gaussian_beta2_divergence_matrix(
        h_grid=perron_comparison_h_grid,
        target_sigma=1.0,
    )
)
perron_comparison_population_vector = perron_vector_from_divergence(
    perron_comparison_population_divergence, perron_comparison_lambda
)

perron_comparison_first_order_vector = first_order_perron_vector(
    divergence_matrix=perron_comparison_population_divergence,
    lambda_value=perron_comparison_lambda,
)
perron_comparison_row_sums = (
    perron_comparison_population_divergence.sum(axis=1)
)
perron_comparison_implicit_index = int(
    np.argmin(perron_comparison_row_sums)
)

perron_comparison_vectors = {
    "sample FFT": perron_comparison_sample_vector,
    "exact population": perron_comparison_population_vector,
    "first-order population": perron_comparison_first_order_vector,
}
perron_comparison_selected_indices = {
    name: int(perron_peak_index(vector))
    for name, vector in perron_comparison_vectors.items()
}

for name, selected_index in perron_comparison_selected_indices.items():                             ### REPORTING
    print(                                                                                          ### REPORTING
        f"{name}: index={selected_index}, "                                                         ### REPORTING
        f"h={perron_comparison_h_grid[selected_index]:.6g}"                                         ### REPORTING
    )                                                                                               ### REPORTING
print(                                                                                              ### REPORTING
    "implicit row-sum characterization: "                                                           ### REPORTING
    f"index={perron_comparison_implicit_index}, "                                                   ### REPORTING
    f"h={perron_comparison_h_grid[perron_comparison_implicit_index]:.6g}"                           ### REPORTING
)                                                                                                   ### REPORTING
print(                                                                                              ### REPORTING
    "maximum population first-order error: "                                                        ### REPORTING
    f"{np.max(np.abs(perron_comparison_population_vector - perron_comparison_first_order_vector)):.3e}"  ### REPORTING
)                                                                                                   ### REPORTING

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5))                                                           ### PLOTTING

for name, vector in perron_comparison_vectors.items():
    relative_vector = perron_comparison_N * vector
    plot_perron_vector(                                                                             ### PLOTTING
        h_grid=perron_comparison_h_grid,                                                            ### PLOTTING
        vector=relative_vector,                                                                     ### PLOTTING
        ax=ax,                                                                                      ### PLOTTING
        label=name,                                                                                 ### PLOTTING
    )                                                                                               ### PLOTTING

ax.axhline(                                                                                         ### PLOTTING
    1.0,                                                                                            ### PLOTTING
    color="#777777",                                                                                ### PLOTTING
    linestyle=":",                                                                                  ### PLOTTING
    label="uniform vector",                                                                         ### PLOTTING
)                                                                                                   ### PLOTTING
ax.set_ylabel("Relative Perron component N v(h) (uniform = 1)")                                     ### PLOTTING
ax.legend()                                                                                         ### PLOTTING
plt.tight_layout()                                                                                  ### PLOTTING
plt.show()                                                                                          ### PLOTTING
